# Лабораторная работа №3

## Задание

Провести классификацию найденного датасета, методами линеной и логистической регрессий

In [1]:
# Импортируем необходимые библиотеки

# Pandas используется для работы с данными в табличном формате
import pandas as pd

# Импортируем функции для разделения данных на обучающую и тестовую выборки, а также для поиска гиперпараметров модели
from sklearn.model_selection import train_test_split, GridSearchCV

# Импортируем классы для кодирования категориальных данных и стандартизации признаков
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Импортируем функции для оценки качества модели
from sklearn.metrics import classification_report, accuracy_score

# Импортируем классы для логистической и линейной регрессии
from sklearn.linear_model import LogisticRegression, LinearRegression

# Импортируем функцию для вычисления среднеквадратичной ошибки (MSE)
from sklearn.metrics import mean_squared_error

In [2]:
# Читаем данные из CSV-файла 'weather.csv' и создаем DataFrame с помощью pandas
df = pd.read_csv('weather.csv', encoding='utf-8')

# Преобразуем значения столбца 'RainToday' из строк 'Yes' и 'No' в логические значения True и False соответственно
df['RainToday'] = df['RainToday'].map({'Yes': True, 'No': False}).astype(bool)

# Преобразуем значения столбца 'RainTomorrow' из строк 'Yes' и 'No' в логические значения True и False соответственно
df['RainTomorrow'] = df['RainTomorrow'].map({'Yes': True, 'No': False}).astype(bool)

# Выводим первые 5 строк DataFrame для ознакомления с данными
print(df.head(5))

# Выводим типы данных каждого столбца DataFrame
print(df.dtypes)

         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   
2  2008-12-03   Albury     12.9     25.7       0.0          NaN       NaN   
3  2008-12-04   Albury      9.2     28.0       0.0          NaN       NaN   
4  2008-12-05   Albury     17.5     32.3       1.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir9am  ... Humidity9am  Humidity3pm  \
0           W           44.0          W  ...        71.0         22.0   
1         WNW           44.0        NNW  ...        44.0         25.0   
2         WSW           46.0          W  ...        38.0         30.0   
3          NE           24.0         SE  ...        45.0         16.0   
4           W           41.0        ENE  ...        82.0         33.0   

   Pressure9am  Pressure3pm  Cloud9am  Cloud3pm  Temp9am  Temp3pm  RainToday  \
0       1007.7    

In [3]:
# Определяем список столбцов, которые нужно удалить из DataFrame
columns_to_drop = ['Humidity3pm', 'Pressure3pm', 'Cloud3pm', 'Temp3pm', 'WindDir9am']

# Удаляем указанные столбцы из DataFrame
df = df.drop(columns=columns_to_drop, axis=1)

# Выводим первые 5 строк обновленного DataFrame для проверки изменений
print(df.head(5))

         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   
2  2008-12-03   Albury     12.9     25.7       0.0          NaN       NaN   
3  2008-12-04   Albury      9.2     28.0       0.0          NaN       NaN   
4  2008-12-05   Albury     17.5     32.3       1.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir3pm  WindSpeed9am  WindSpeed3pm  \
0           W           44.0        WNW          20.0          24.0   
1         WNW           44.0        WSW           4.0          22.0   
2         WSW           46.0        WSW          19.0          26.0   
3          NE           24.0          E          11.0           9.0   
4           W           41.0         NW           7.0          20.0   

   Humidity9am  Pressure9am  Cloud9am  Temp9am  RainToday  RainTomorrow  
0         71.0       1007.7       8.

In [4]:
# Удаляем строки с отсутствующими значениями из DataFrame
df.dropna(inplace=True)

In [5]:
# Создаем словарь для хранения объектов LabelEncoder для каждого категориального признака
label_encoders = {}

# Проходим по всем столбцам DataFrame, которые имеют тип данных 'object' (категориальные признаки)
for column in df.select_dtypes(include=['object']).columns:
    # Создаем новый LabelEncoder для текущего столбца
    label_encoders[column] = LabelEncoder()
    
    # Преобразуем категориальные значения в числовые и сохраняем их обратно в DataFrame
    df[column] = label_encoders[column].fit_transform(df[column])

# Выводим первые 5 строк DataFrame для проверки изменений
print(df.head(5))

      Date  Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
6049   418         4     17.9     35.2       0.0         12.0      12.3   
6050   419         4     18.4     28.9       0.0         14.8      13.0   
6052   421         4     19.4     37.6       0.0         10.8      10.6   
6053   422         4     21.9     38.4       0.0         11.4      12.2   
6054   423         4     24.2     41.0       0.0         11.2       8.4   

      WindGustDir  WindGustSpeed  WindDir3pm  WindSpeed9am  WindSpeed3pm  \
6049           11           48.0          12           6.0          20.0   
6050            8           37.0          10          19.0          19.0   
6052            5           46.0           6          30.0          15.0   
6053           14           31.0          15           6.0           6.0   
6054           14           35.0          14          17.0          13.0   

      Humidity9am  Pressure9am  Cloud9am  Temp9am  RainToday  RainTomorrow  
6049         20

In [6]:
# Создаем объект StandardScaler для масштабирования числовых признаков
scaler = StandardScaler()

# Выбираем все столбцы, которые имеют числовые типы данных (int32, int64, float32, float64)
numeric_features = df.select_dtypes(include=['int32', 'int64', 'float32', 'float64']).columns

# Применяем масштабирование ко всем числовым признакам и сохраняем их обратно в DataFrame
df[numeric_features] = scaler.fit_transform(df[numeric_features])

# Выводим первые 5 строк DataFrame для проверки изменений
print(df.head(5))

          Date  Location   MinTemp   MaxTemp  Rainfall  Evaporation  Sunshine  \
6049 -1.545272 -1.182653  0.716750  1.589217 -0.302062     1.771778  1.217662   
6050 -1.544105 -1.182653  0.793979  0.686505 -0.302062     2.529392  1.404005   
6052 -1.541771 -1.182653  0.948437  1.933107 -0.302062     1.447086  0.765115   
6053 -1.540604 -1.182653  1.334582  2.047737 -0.302062     1.609432  1.191042   
6054 -1.539437 -1.182653  1.689834  2.420285 -0.302062     1.555317  0.179465   

      WindGustDir  WindGustSpeed  WindDir3pm  WindSpeed9am  WindSpeed3pm  \
6049     0.728337       0.553696    0.933445     -1.077916      0.043157   
6050     0.102162      -0.269995    0.509257      0.435573     -0.074484   
6052    -0.524013       0.403934   -0.339118      1.716217     -0.545047   
6053     1.354511      -0.719280    1.569726     -1.077916     -1.603814   
6054     1.354511      -0.419757    1.357632      0.202728     -0.780329   

      Humidity9am  Pressure9am  Cloud9am   Temp9am  Rain

In [7]:
# Разделяем DataFrame на матрицу признаков X и вектор целевой переменной Y
X = df.drop('RainTomorrow', axis=1)  # Удаляем столбец 'RainTomorrow' и сохраняем все остальные столбцы в X
Y = df['RainTomorrow']               # Сохраняем столбец 'RainTomorrow' в качестве целевой переменной Y

# Разделяем данные на обучающий и тестовый наборы
# test_size=0.2 означает, что 20% данных будет использовано для тестирования, а 80% для обучения
# random_state=42 используется для воспроизводимости результатов
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [8]:
# Обучение модели линейной регрессии
linear_model = LinearRegression()

In [9]:
# Определяем сетку параметров для поиска оптимальных параметров модели линейной регрессии
param_grid_linear = {
    'fit_intercept': [True, False],  # Параметр, указывающий, должен ли модель вычислять перехват (смещение)
    'copy_X': [True, False],          # Параметр, указывающий, следует ли копировать матрицу X (при использовании inplace-операций)
    'positive': [True, False],        # Параметр, указывающий, должны ли коэффициенты быть неотрицательными
}

In [10]:
# Поиск оптимальных параметров с использованием кросс-валидации
grid_search_linear = GridSearchCV(linear_model, param_grid_linear, cv=5)
grid_search_linear.fit(X_train, Y_train)

GridSearchCV(cv=5, estimator=LinearRegression(),
             param_grid={'copy_X': [True, False],
                         'fit_intercept': [True, False],
                         'positive': [True, False]})

In [11]:
# Получение лучших параметров
best_params_linear = grid_search_linear.best_params_
print("Лучшие параметры:", best_params_linear)

Лучшие параметры: {'copy_X': True, 'fit_intercept': True, 'positive': False}


In [12]:
# Обучение модели с лучшими параметрами
best_svm_linear = grid_search_linear.best_estimator_
best_svm_linear.fit(X_train, Y_train)

LinearRegression()

In [13]:
# Предсказание на тестовом наборе данных
Y_pred = best_svm_linear.predict(X_test)

In [14]:
# Оценка модели на тестовом наборе данных
mse_linear = mean_squared_error(Y_test, Y_pred)
print("Среднеквадратичная ошибка модели:", mse_linear)

Среднеквадратичная ошибка модели: 0.1147552409942324


In [15]:
# Создаем объект LogisticRegression для обучения модели логистической регрессии
logistic_model = LogisticRegression()

In [16]:
# Определение сетки параметров
param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear'],
}

In [17]:
# Получение лучших параметров
best_params_linear = grid_search_linear.best_params_
print("Лучшие параметры:", best_params_linear)

GridSearchCV(cv=5, estimator=LogisticRegression(),
             param_grid={'C': [0.1, 1, 10], 'penalty': ['l2'],
                         'solver': ['lbfgs', 'liblinear']})

In [18]:
# Получение лучших параметров
best_params = grid_search.best_params_
print("Лучшие параметры:", best_params)

Лучшие параметры: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}


In [19]:
# Обучение модели с лучшими параметрами
best_svm = grid_search.best_estimator_
best_svm.fit(X_train, Y_train)

LogisticRegression(C=10, solver='liblinear')

In [20]:
# Предсказание на тестовом наборе данных
Y_pred = best_svm.predict(X_test)

In [21]:
# Оценка модели на тестовом наборе данных
accuracy = accuracy_score(Y_test, Y_pred)
print("Оценка модели:", accuracy)

Оценка модели: 0.8433067561896769


In [22]:
# Оценка модели логистической регрессии
report_log = classification_report(Y_test, Y_pred)
print(report_log)

              precision    recall  f1-score   support

       False       0.87      0.95      0.90      9326
        True       0.71      0.47      0.57      2589

    accuracy                           0.84     11915
   macro avg       0.79      0.71      0.74     11915
weighted avg       0.83      0.84      0.83     11915

